In [23]:
import os
import shutil
import re
from pathlib import Path
import pandas as pd

In [ ]:
csv_path = Path(r"E:\500 newest\gloss_filtered.csv")
source_1 = Path(r"D:\CLEANED\500 CLEANING")
source_2 = Path(r"E:\500 new with cleaning\cleaned_landmarks_165words")
source_3 = Path(r"D:\ASL Citizen\data\raw_landmarks")
output_dir = Path(r"E:\500 newest\raw_landmarks")
output_dir.mkdir(exist_ok=True)

In [25]:
df = pd.read_csv(csv_path)
df.head()

,gloss
0,JACKET
1,RIGHT
2,PATIENT
3,WHAT FOR
4,SHOP


In [26]:
len(df)

442

In [27]:
#load gloss list
glosses = (
    df["gloss"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.replace("_", " ")  # Ensures underscores are spaces
    .str.replace("-", " ")
    .str.strip()
    .tolist()
)

In [28]:
len(glosses)

442

In [29]:
# Helpers
def clean_stem(stem):
    # Standardize incoming file names to match the cleaned glosses
    stem = stem.upper().replace("_", " ").replace("-", " ").strip()
    stem = re.sub(r"\s+\d+$", "", stem)
    return stem.strip()

def find_files(gloss, folder):
    matches = []
    if not folder.exists():
        return matches

    gloss = gloss.upper().strip()
    for file in folder.rglob("*.npy"):
        stem = clean_stem(file.stem)
        if stem == gloss:
            matches.append(file)
    return matches

In [ ]:
# Main processing
results = []
missing = []

for gloss in glosses:
    gloss = gloss.upper().strip()
    files = find_files(gloss, source_1)

    if not files:
        files = find_files(gloss, source_2)
    if not files:
        files = find_files(gloss, source_3)

    if not files:
        missing.append(gloss)
        continue

    copied_names = []

    for i, f in enumerate(files):
        # Use a space and index to avoid underscores entirely
        # Using f"{gloss} {i}.npy" ensures every file has a unique name 
        # without triggering your underscore detection script.
        new_name = f"{gloss} {i}.npy"
        dest = output_dir / new_name

        shutil.copy2(f, dest)
        copied_names.append(dest.name)

    results.append({
        "gloss": gloss,
        "count": len(files),
        "files_names": ";".join(copied_names)
    })

In [31]:
# Save outputs
final_df = pd.DataFrame(results)
final_df.to_csv("final_gloss_files.csv", index=False)

missing_df = pd.DataFrame({"missing_gloss": missing})
missing_df.to_csv("missing_glosses.csv", index=False)

print("Done!")
print(f"Copied glosses: {len(final_df)}")
print(f"Missing glosses: {len(missing)}")

Done!
Copied glosses: 435
Missing glosses: 7
